# ML69_K1 — Food Delivery App Customer Segmentation (Rank #1 Solution: ARI = 0.93635)

**Team:** IdeaMee  
**Author / Branch:** Taer (`Nisaranun`)  
**Evaluation Metric:** Adjusted Rand Index (ARI) — Public/Final Score: **`0.93635` (#1 Leaderboard)**

---
### สรุปแนวคิดหลักของโมเดลที่ชนะอันดับ 1
1. **Feature Selection:** คัดเลือกเฉพาะ 15 ฟีเจอร์พฤติกรรมการสั่งอาหาร + `membership_tier` (ตัดฟีเจอร์ตัวเครื่อง/ระบบ 10 ตัว, `os`, `district`, `id` ที่เป็น Noise ออก)
2. **Exact Inverse-Generator Transform:** แปลงข้อมูลให้กลับเป็น Multivariate Normal สมบูรณ์ (`skew ≈ 0`, `kurtosis ≈ 0`, Mahalanobis $\chi^2(15)$ ตรงทฤษฎี):
   - ฟีเจอร์จำนวนบวก 6 ตัว (`orders_per_month`, `avg_order_value_thb`, `spend_per_month_thb`, `app_sessions_per_month`, `avg_delivery_km`, `days_since_last_order`) $ightarrow \ln(x)$
   - ฟีเจอร์สัดส่วน `0–1` ทั้ง 7 ตัว (`night_order_share`, `discount_order_share`, `weekend_order_share`, `share_*`) $ightarrow 	ext{logit}(x) = \ln\!\left(rac{x}{1-x}ight)$
   - `avg_items_per_order` $ightarrow \ln(x - 1.0)$
3. **Exact Marginalized & Censored Tobit Joint EM (`k = 7`):**
   - จัดการค่าว่างของ `avg_rating_given` (367 แถว) ด้วย **Exact Marginalization** ไม่เดาค่ามาหลอกโมเดล
   - จัดการค่าที่ชนเพดาน (`days_since_last_order = 180.0`, `avg_rating_given = 5.0`) ด้วย **Tobit Censored Likelihood & Inverse Mills Ratio M-step**
   - เทรน `membership_tier` (Multinomial) และ `is_missing` (Bernoulli) ร่วมใน EM
4. **Bayesian Log-Posterior Ensemble (PURE MAP):**
   - รวมพลัง **70% 14D 5-Model Sparse Tobit EM (`Adaptive-Z` & `Smooth-SCAD`)** + **30% 15D Full Tobit EM** และตัดสินใจด้วย **PURE `MAP` (`argmax`)**


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import multivariate_normal, norm
from scipy.special import logsumexp
from scipy.optimize import linear_sum_assignment
from sklearn.impute import KNNImputer
from sklearn.mixture import GaussianMixture
from pathlib import Path


## 1. โหลดข้อมูลและแปลงฟีเจอร์ด้วย Exact Inverse-Generator Transform


In [ ]:
DATA_PATH = Path('data.csv')
df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)

log_cols_14 = [
    'orders_per_month', 'avg_order_value_thb', 'app_sessions_per_month',
    'avg_delivery_km', 'days_since_last_order'
]
logit_cols = [
    'night_order_share', 'discount_order_share', 'weekend_order_share',
    'share_thai_food', 'share_japanese_korean', 'share_fastfood_western',
    'share_drinks_dessert'
]

def build_features(df, include_spend=False):
    X = pd.DataFrame(index=df.index)
    for c in log_cols_14:
        X[c] = np.log(df[c].values)
    if include_spend:
        X['spend_per_month_thb'] = np.log(df['spend_per_month_thb'].values)
    for c in logit_cols:
        X[c] = np.log(df[c].values / (1.0 - df[c].values))
    X['avg_items_per_order'] = np.log(np.maximum(df['avg_items_per_order'].values - 1.0, 1e-4))
    X['avg_rating_given'] = df['avg_rating_given'].values  # Keep NaN for exact marginalization
    cols = list(X.columns)
    vals = X.values
    X_std = (vals - np.nanmean(vals, axis=0)) / np.nanstd(vals, axis=0)
    return X_std, cols

X14_s, cols14 = build_features(df, include_spend=False)
X15_s, cols15 = build_features(df, include_spend=True)

tier_map = {'none': 0, 'silver': 1, 'gold': 2, 'platinum': 3}
tier_idx = df['membership_tier'].map(tier_map).values
days_cens = (df['days_since_last_order'] == 180.0).values
rate_cens = (df['avg_rating_given'] == 5.0).values
print('Prepared 14D:', X14_s.shape, '| 15D:', X15_s.shape)


## 2. Exact Marginalized & Censored Tobit Joint EM (`Adaptive-Z` / `Smooth-SCAD` / `Full`)


In [ ]:
def fit_exact_tobit_em(X_tr, cols, t_tr, dc_tr, rc_tr, mode='scad', param=(2.5, 4.5), K=7, max_iter=200, seed=42):
    N_tr, D = X_tr.shape
    days_idx = cols.index('days_since_last_order')
    rate_idx = cols.index('avg_rating_given')
    obs_tr = ~np.isnan(X_tr[:, rate_idx])
    miss_tr = ~obs_tr
    idx_no_r = [j for j in range(D) if j != rate_idx]

    X_imp = KNNImputer(n_neighbors=15).fit_transform(X_tr)
    gm0 = GaussianMixture(K, covariance_type='full', n_init=25, random_state=seed).fit(X_imp)
    resp = gm0.predict_proba(X_imp)

    def e_step(pi, p_tier, p_miss, mu, Sigma):
        lp = np.zeros((N_tr, K))
        for k in range(K):
            lp[obs_tr, k] = multivariate_normal.logpdf(X_tr[obs_tr], mean=mu[k], cov=Sigma[k]) + np.log(1.0 - p_miss[k])
            lp[miss_tr, k] = multivariate_normal.logpdf(
                X_tr[np.ix_(miss_tr, idx_no_r)], mean=mu[k, idx_no_r], cov=Sigma[k][np.ix_(idx_no_r, idx_no_r)]
            ) + np.log(p_miss[k])
            # Right-censored days_since_last_order == 180.0
            for mask_c, use_cols in [(dc_tr & obs_tr, list(range(D))), (dc_tr & miss_tr, idx_no_r)]:
                if not mask_c.any(): continue
                other = [j for j in use_cols if j != days_idx]
                c_val = X_tr[mask_c, days_idx]
                x_o = X_tr[np.ix_(mask_c, other)]
                S_oo = Sigma[k][np.ix_(other, other)]
                S_do = Sigma[k][days_idx, other]
                S_dd = Sigma[k][days_idx, days_idx]
                inv_Soo = np.linalg.inv(S_oo)
                mu_c = mu[k, days_idx] + (x_o - mu[k, other]) @ (inv_Soo @ S_do)
                sd_c = np.sqrt(max(S_dd - S_do @ inv_Soo @ S_do, 1e-6))
                lp[mask_c, k] += norm.logsf(c_val, loc=mu_c, scale=sd_c) - norm.logpdf(c_val, loc=mu_c, scale=sd_c)
            # Right-censored avg_rating_given == 5.0
            if rc_tr.any():
                other = idx_no_r
                c_val = X_tr[rc_tr, rate_idx]
                x_o = X_tr[np.ix_(rc_tr, other)]
                S_oo = Sigma[k][np.ix_(other, other)]
                S_ro = Sigma[k][rate_idx, other]
                S_rr = Sigma[k][rate_idx, rate_idx]
                inv_Soo = np.linalg.inv(S_oo)
                mu_c = mu[k, rate_idx] + (x_o - mu[k, other]) @ (inv_Soo @ S_ro)
                sd_c = np.sqrt(max(S_rr - S_ro @ inv_Soo @ S_ro, 1e-6))
                lp[rc_tr, k] += norm.logsf(c_val, loc=mu_c, scale=sd_c) - norm.logpdf(c_val, loc=mu_c, scale=sd_c)
            lp[:, k] += np.log(pi[k]) + np.log(p_tier[k, t_tr])
        ln = logsumexp(lp, axis=1, keepdims=True)
        return np.exp(lp - ln), ln.sum()

    mu = np.zeros((K, D))
    Sigma = np.array([np.eye(D) for _ in range(K)])
    prev_ll = -np.inf

    for it in range(max_iter):
        Nk = resp.sum(axis=0) + 1e-10
        pi = Nk / N_tr
        p_tier = np.zeros((K, 4))
        for t in range(4):
            p_tier[:, t] = (resp[t_tr == t].sum(axis=0) + 1e-3) / (Nk + 4e-3)
        p_miss = (resp[miss_tr].sum(axis=0) + 1e-3) / (Nk + 2e-3)

        for k in range(K):
            w = resp[:, k]
            Xk = X_tr.copy()
            extra_var = np.zeros(D)
            if it > 0:
                if miss_tr.any():
                    S_oo = Sigma[k][np.ix_(idx_no_r, idx_no_r)]
                    S_ro = Sigma[k][rate_idx, idx_no_r]
                    inv_Soo = np.linalg.inv(S_oo)
                    Xk[miss_tr, rate_idx] = mu[k, rate_idx] + (X_tr[np.ix_(miss_tr, idx_no_r)] - mu[k, idx_no_r]) @ (inv_Soo @ S_ro)
                    var_r_cond = max(Sigma[k][rate_idx, rate_idx] - S_ro @ inv_Soo @ S_ro, 1e-6)
                    extra_var[rate_idx] += (w[miss_tr].sum() / Nk[k]) * var_r_cond
                if dc_tr.any():
                    other = [j for j in range(D) if j != days_idx]
                    c_val = Xk[dc_tr, days_idx]
                    S_oo = Sigma[k][np.ix_(other, other)]
                    S_do = Sigma[k][days_idx, other]
                    S_dd = Sigma[k][days_idx, days_idx]
                    inv_Soo = np.linalg.inv(S_oo)
                    mu_c = mu[k, days_idx] + (Xk[np.ix_(dc_tr, other)] - mu[k, other]) @ (inv_Soo @ S_do)
                    var_c = max(S_dd - S_do @ inv_Soo @ S_do, 1e-6)
                    sd_c = np.sqrt(var_c)
                    alpha = (c_val - mu_c) / sd_c
                    mills = np.exp(norm.logpdf(alpha) - norm.logsf(alpha))
                    Xk[dc_tr, days_idx] = mu_c + sd_c * mills
                    extra_var[days_idx] += (w[dc_tr] * np.maximum(var_c * (1.0 + alpha * mills - mills**2), 0.0)).sum() / Nk[k]
                if rc_tr.any():
                    other = idx_no_r
                    c_val = Xk[rc_tr, rate_idx]
                    S_oo = Sigma[k][np.ix_(other, other)]
                    S_ro = Sigma[k][rate_idx, other]
                    S_rr = Sigma[k][rate_idx, rate_idx]
                    inv_Soo = np.linalg.inv(S_oo)
                    mu_c = mu[k, rate_idx] + (Xk[np.ix_(rc_tr, other)] - mu[k, other]) @ (inv_Soo @ S_ro)
                    var_c = max(S_rr - S_ro @ inv_Soo @ S_ro, 1e-6)
                    sd_c = np.sqrt(var_c)
                    alpha = (c_val - mu_c) / sd_c
                    mills = np.exp(norm.logpdf(alpha) - norm.logsf(alpha))
                    Xk[rc_tr, rate_idx] = mu_c + sd_c * mills
                    extra_var[rate_idx] += (w[rc_tr] * np.maximum(var_c * (1.0 + alpha * mills - mills**2), 0.0)).sum() / Nk[k]
                mu[k] = (w[:, None] * Xk).sum(axis=0) / Nk[k]
                diff = Xk - mu[k]
                S = (diff.T * w) @ diff / Nk[k] + np.diag(extra_var)
            else:
                for j in range(D):
                    if j == rate_idx:
                        w_o = w[obs_tr]
                        mu[k, j] = (w_o * X_tr[obs_tr, j]).sum() / (w_o.sum() + 1e-10)
                    else:
                        mu[k, j] = (w * X_tr[:, j]).sum() / Nk[k]
                diff_o = X_tr[obs_tr] - mu[k]
                S = (diff_o.T * w[obs_tr]) @ diff_o / (w[obs_tr].sum() + 1e-10)

            S = 0.5 * (S + S.T) + np.eye(D) * 1e-5
            if mode != 'none':
                sd = np.sqrt(np.maximum(np.diag(S), 1e-6))
                R = S / np.outer(sd, sd)
                if mode == 'hard_z':
                    th = param / np.sqrt(Nk[k])
                    R = R * ((np.abs(R) >= th) | np.eye(D, dtype=bool))
                elif mode == 'scad':
                    z_lo, z_hi = param
                    t_lo, t_hi = z_lo / np.sqrt(Nk[k]), z_hi / np.sqrt(Nk[k])
                    w_scad = np.clip((np.abs(R) - t_lo) / (t_hi - t_lo), 0.0, 1.0)
                    np.fill_diagonal(w_scad, 1.0)
                    R = R * w_scad
                S = R * np.outer(sd, sd)

            S = 0.5 * (S + S.T) + np.eye(D) * 1e-5
            ev, evec = np.linalg.eigh(S)
            M = (evec * np.maximum(ev, 1e-4)) @ evec.T
            Sigma[k] = 0.5 * (M + M.T)

        resp, ll = e_step(pi, p_tier, p_miss, mu, Sigma)
        if abs(ll - prev_ll) < 1e-5:
            break
        prev_ll = ll
    return resp, ll

def align_proba(P_ref, P_other):
    cm = P_ref.T @ P_other
    _, col_ind = linear_sum_assignment(-cm)
    return P_other[:, col_ind]


## 3. เทรน Bayesian Log-Posterior Ensemble (`70% 14D Sparse + 30% 15D Full`, PURE `MAP`) และบันทึก `submission.csv`


In [ ]:
P_ref, _ = fit_exact_tobit_em(X14_s, cols14, tier_idx, days_cens, rate_cens, mode='hard_z', param=3.5)
models_14d = [
    P_ref,
    align_proba(P_ref, fit_exact_tobit_em(X14_s, cols14, tier_idx, days_cens, rate_cens, mode='hard_z', param=4.0)[0]),
    align_proba(P_ref, fit_exact_tobit_em(X14_s, cols14, tier_idx, days_cens, rate_cens, mode='scad', param=(2.5, 4.5))[0]),
    align_proba(P_ref, fit_exact_tobit_em(X14_s, cols14, tier_idx, days_cens, rate_cens, mode='scad', param=(3.0, 4.5))[0]),
    align_proba(P_ref, fit_exact_tobit_em(X14_s, cols14, tier_idx, days_cens, rate_cens, mode='scad', param=(3.0, 5.0))[0]),
]
P_15_full = align_proba(P_ref, fit_exact_tobit_em(X15_s, cols15, tier_idx, days_cens, rate_cens, mode='none', param=0.0)[0])

logP_14 = np.mean([np.log(np.clip(P, 1e-12, 1.0)) for P in models_14d], axis=0)
logP_14 -= logsumexp(logP_14, axis=1, keepdims=True)
logP_15 = np.log(np.clip(P_15_full, 1e-12, 1.0))

# Winning Rank #1 Blend (0.93635): 70% 14D Sparse + 30% 15D Full, PURE MAP
logP_final = 0.70 * logP_14 + 0.30 * logP_15
clusters = logP_final.argmax(axis=1)

sub = pd.DataFrame({'id': df['id'].astype(int), 'cluster': clusters.astype(int)})
sub.to_csv('submission.csv', index=False)
print('Saved submission.csv! Shape:', sub.shape)
print('Cluster counts:\n', sub['cluster'].value_counts().sort_index())
sub.head(10)
